# 01 - What is a Neural Network?

---

## Learning Objectives

By the end of this notebook, you will be able to:

- Define key neural network components: **neuron**, **layers**, **weights**, **bias**, and **activation**
- Explain the analogy between biological and artificial neurons
- Understand the **Universal Approximation Theorem** at an intuitive level
- Implement a single neuron from scratch in NumPy
- Stack neurons to form a layer and understand how layers compose into a network

## Prerequisites

- Basic Python programming (functions, classes, NumPy arrays)
- High-school linear algebra (dot products, matrix multiplication)
- Familiarity with matplotlib for plotting

## Table of Contents

1. [What is a Neuron?](#1-what-is-a-neuron)
2. [Biological vs Artificial Neuron](#2-biological-vs-artificial-neuron)
3. [Layers: Input, Hidden, Output](#3-layers-input-hidden-output)
4. [Weights, Bias, and Activation](#4-weights-bias-and-activation)
5. [Visualizing a Simple Network](#5-visualizing-a-simple-network)
6. [Universal Approximation Theorem](#6-universal-approximation-theorem)
7. [Code: A Single Neuron in NumPy](#7-code-a-single-neuron-in-numpy)
8. [Code: Stacking Neurons into a Layer](#8-code-stacking-neurons-into-a-layer)
9. [Exercise: Different Activations](#9-exercise-different-activations)
10. [Common Mistakes & Debugging Tips](#10-common-mistakes--debugging-tips)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

np.random.seed(42)

%matplotlib inline
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['figure.dpi'] = 100

---
## 1. What is a Neuron?

A **neuron** (also called a **unit** or **node**) is the fundamental building block of a neural network. It performs three simple operations:

1. **Receives inputs** $x_1, x_2, \dots, x_n$
2. **Computes a weighted sum** plus a bias:
   $$z = w_1 x_1 + w_2 x_2 + \cdots + w_n x_n + b = \mathbf{w}^T \mathbf{x} + b$$
3. **Applies an activation function** to produce the output:
   $$a = \sigma(z)$$

That is it. Every neural network, no matter how large, is built from neurons doing exactly these three steps.

---
## 2. Biological vs Artificial Neuron

| Biological Neuron | Artificial Neuron |
|---|---|
| **Dendrites** receive signals from other neurons | **Inputs** $x_1, x_2, \dots$ arrive from data or previous layer |
| **Synapse strength** modulates signal | **Weights** $w_i$ scale each input |
| **Cell body (soma)** integrates signals | **Weighted sum** $z = \mathbf{w}^T\mathbf{x} + b$ |
| **Threshold / firing** (all-or-nothing action potential) | **Activation function** $\sigma(z)$ |
| **Axon** transmits output to other neurons | **Output** $a$ feeds into next layer |

**Important caveat:** The analogy is *loose*. Biological neurons are far more complex (timing, chemical signaling, dendritic computation). Artificial neurons are a simplified mathematical abstraction that turns out to be extremely useful.

---
## 3. Layers: Input, Hidden, Output

Neurons are organized into **layers**:

- **Input layer:** Holds the raw features. No computation happens here -- it simply passes data forward. If your data has $d$ features, the input layer has $d$ nodes.
- **Hidden layer(s):** Where the actual computation and feature extraction happens. Each neuron in a hidden layer applies weights, bias, and an activation. Networks can have one or many hidden layers ("deep" = multiple hidden layers).
- **Output layer:** Produces the final prediction. Its size and activation depend on the task:
  - Regression: 1 neuron, linear activation
  - Binary classification: 1 neuron, sigmoid activation
  - Multi-class classification: $K$ neurons, softmax activation

---
## 4. Weights, Bias, and Activation

**Weights** ($\mathbf{w}$): Learnable parameters that determine how much influence each input has. During training, these are adjusted to minimize the loss function.

**Bias** ($b$): A learnable scalar that shifts the activation. Without bias, the neuron's decision boundary would always pass through the origin.

**Activation function** ($\sigma$): A (usually nonlinear) function applied after the weighted sum. Without nonlinearity, stacking layers would be equivalent to a single linear transformation, making deep networks pointless.

Common activation functions (covered in detail in Notebook 02):
- **Sigmoid:** $\sigma(z) = \frac{1}{1 + e^{-z}}$ -- squashes output to (0, 1)
- **ReLU:** $\text{ReLU}(z) = \max(0, z)$ -- most popular in hidden layers
- **Tanh:** $\tanh(z)$ -- squashes output to (-1, 1)

---
## 5. Visualizing a Simple Network

Below we draw a simple fully-connected (dense) neural network with:
- 3 input neurons
- 4 hidden neurons
- 2 output neurons

In [ ]:
def draw_neural_network(layer_sizes, ax=None):
    """
    Draw a fully-connected neural network diagram.
    
    Parameters
    ----------
    layer_sizes : list of int
        Number of neurons in each layer, e.g. [3, 4, 2].
    ax : matplotlib Axes, optional
        Axes to draw on. If None, creates a new figure.
    """
    if ax is None:
        fig, ax = plt.subplots(1, 1, figsize=(10, 6))
    
    n_layers = len(layer_sizes)
    max_neurons = max(layer_sizes)
    
    # Spacing
    h_spacing = 2.0  # horizontal spacing between layers
    v_spacing = 1.2  # vertical spacing between neurons
    radius = 0.3
    
    # Compute positions for each neuron
    positions = []  # positions[layer][neuron] = (x, y)
    for i, size in enumerate(layer_sizes):
        layer_positions = []
        x = i * h_spacing
        y_start = (max_neurons - size) * v_spacing / 2  # center vertically
        for j in range(size):
            y = y_start + j * v_spacing
            layer_positions.append((x, y))
        positions.append(layer_positions)
    
    # Draw connections (edges) between consecutive layers
    for i in range(n_layers - 1):
        for (x1, y1) in positions[i]:
            for (x2, y2) in positions[i + 1]:
                ax.plot([x1, x2], [y1, y2], 'b-', alpha=0.2, linewidth=0.8)
    
    # Draw neurons (circles)
    layer_labels = ['Input\nLayer', 'Hidden\nLayer', 'Output\nLayer']
    if n_layers > 3:
        layer_labels = ['Input'] + [f'Hidden {i}' for i in range(1, n_layers - 1)] + ['Output']
    
    colors = ['#4CAF50', '#2196F3', '#FF9800']  # green, blue, orange
    if n_layers > 3:
        colors = ['#4CAF50'] + ['#2196F3'] * (n_layers - 2) + ['#FF9800']
    
    for i, layer_pos in enumerate(positions):
        color = colors[min(i, len(colors) - 1)]
        for j, (x, y) in enumerate(layer_pos):
            circle = plt.Circle((x, y), radius, color=color, ec='black',
                                linewidth=1.5, zorder=4)
            ax.add_patch(circle)
            # Label neuron
            if i == 0:
                ax.text(x, y, f'$x_{j+1}$', ha='center', va='center',
                        fontsize=11, fontweight='bold', zorder=5)
            elif i == n_layers - 1:
                ax.text(x, y, f'$y_{j+1}$', ha='center', va='center',
                        fontsize=11, fontweight='bold', zorder=5)
            else:
                ax.text(x, y, f'$h_{j+1}$', ha='center', va='center',
                        fontsize=11, fontweight='bold', zorder=5)
        
        # Layer label below
        x_center = positions[i][0][0]
        y_bottom = min(p[1] for p in positions[i]) - 0.9
        label = layer_labels[min(i, len(layer_labels) - 1)]
        ax.text(x_center, y_bottom, label, ha='center', va='center',
                fontsize=10, fontweight='bold')
    
    ax.set_xlim(-0.8, (n_layers - 1) * h_spacing + 0.8)
    ax.set_ylim(-1.5, max_neurons * v_spacing)
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title('Fully-Connected Neural Network', fontsize=14, fontweight='bold')
    
    return ax


fig, ax = plt.subplots(1, 1, figsize=(10, 6))
draw_neural_network([3, 4, 2], ax=ax)
plt.tight_layout()
plt.show()

**Diagram explanation:**
- **Green circles** = input layer (3 features: $x_1, x_2, x_3$)
- **Blue circles** = hidden layer (4 neurons: $h_1, h_2, h_3, h_4$)
- **Orange circles** = output layer (2 outputs: $y_1, y_2$)
- **Lines** = weighted connections (each carries a learnable weight $w_{ij}$)

Every neuron in one layer is connected to every neuron in the next layer -- hence the name **fully-connected** (or **dense**) layer.

---
## 6. Universal Approximation Theorem

**Theorem (informal):** A feedforward neural network with a single hidden layer containing a finite number of neurons can approximate any continuous function on a compact subset of $\mathbb{R}^n$, given a suitable activation function (e.g., sigmoid, ReLU).

**Intuition:**
- Each neuron in the hidden layer can learn a "building block" -- a small piece of the target function.
- With enough neurons, you can combine these pieces to approximate any smooth function to arbitrary precision.
- Think of it like LEGO bricks: with enough small pieces, you can build any shape.

**Important caveats:**
- The theorem says such a network *exists*, not that gradient descent will *find* it.
- It says nothing about how many neurons you need (could be astronomically many for a single hidden layer).
- In practice, **deeper** networks (multiple hidden layers) tend to be far more parameter-efficient than a single very wide layer.

This theorem is why neural networks are so powerful -- they are universal function approximators.

---
## 7. Code: A Single Neuron in NumPy

Let us implement a single neuron step by step.

In [ ]:
def sigmoid(z):
    """Sigmoid activation function."""
    return 1.0 / (1.0 + np.exp(-z))


def single_neuron(x, w, b, activation_fn=sigmoid):
    """
    Compute the output of a single neuron.
    
    Parameters
    ----------
    x : np.ndarray, shape (n_features,)
        Input vector.
    w : np.ndarray, shape (n_features,)
        Weight vector.
    b : float
        Bias term.
    activation_fn : callable
        Activation function to apply.
    
    Returns
    -------
    a : float
        Neuron output after activation.
    z : float
        Pre-activation value (weighted sum + bias).
    """
    z = np.dot(w, x) + b  # weighted sum + bias
    a = activation_fn(z)   # apply activation
    return a, z


# Example: a neuron with 3 inputs
np.random.seed(42)
x = np.array([0.5, -0.3, 0.8])         # input features
w = np.random.randn(3)                   # random weights
b = np.random.randn()                    # random bias

a, z = single_neuron(x, w, b)

print(f"Input x:            {x}")
print(f"Weights w:          {w}")
print(f"Bias b:             {b:.4f}")
print(f"Pre-activation z:   {z:.4f}")
print(f"Output a = sig(z):  {a:.4f}")

In [ ]:
# Let's verify step by step
z_manual = w[0]*x[0] + w[1]*x[1] + w[2]*x[2] + b
a_manual = 1.0 / (1.0 + np.exp(-z_manual))

print(f"Manual z: {z_manual:.4f}  (matches: {np.isclose(z, z_manual)})")
print(f"Manual a: {a_manual:.4f}  (matches: {np.isclose(a, a_manual)})")

---
## 8. Code: Stacking Neurons into a Layer

A **layer** is simply multiple neurons operating in parallel on the same input. Instead of computing one dot product, we compute many at once using matrix multiplication.

For a layer with $m$ neurons receiving $n$ inputs:

$$\mathbf{z} = \mathbf{W} \mathbf{x} + \mathbf{b}$$

where:
- $\mathbf{W} \in \mathbb{R}^{m \times n}$ -- weight matrix (each row = one neuron's weights)
- $\mathbf{x} \in \mathbb{R}^{n}$ -- input vector
- $\mathbf{b} \in \mathbb{R}^{m}$ -- bias vector
- $\mathbf{z} \in \mathbb{R}^{m}$ -- pre-activation vector

In [ ]:
def dense_layer(x, W, b, activation_fn=sigmoid):
    """
    Compute the output of a dense (fully-connected) layer.
    
    Parameters
    ----------
    x : np.ndarray, shape (n_inputs,)
        Input vector.
    W : np.ndarray, shape (n_neurons, n_inputs)
        Weight matrix.
    b : np.ndarray, shape (n_neurons,)
        Bias vector.
    activation_fn : callable
        Activation function.
    
    Returns
    -------
    a : np.ndarray, shape (n_neurons,)
        Layer output after activation.
    z : np.ndarray, shape (n_neurons,)
        Pre-activation values.
    """
    z = W @ x + b          # matrix multiply + bias  (m,n)@(n,) + (m,) = (m,)
    a = activation_fn(z)   # element-wise activation
    return a, z


# Example: layer with 4 neurons, 3 inputs
np.random.seed(42)
n_inputs = 3
n_neurons = 4

x = np.array([0.5, -0.3, 0.8])
W = np.random.randn(n_neurons, n_inputs)  # shape (4, 3)
b = np.random.randn(n_neurons)            # shape (4,)

a, z = dense_layer(x, W, b)

print(f"Input x shape:   {x.shape}")
print(f"Weight W shape:  {W.shape}")
print(f"Bias b shape:    {b.shape}")
print(f"Output a shape:  {a.shape}")
print()
print(f"Pre-activation z: {z}")
print(f"Output a:         {a}")

In [ ]:
# Verify: each neuron's output should match calling single_neuron individually
print("Verification -- each row of W is one neuron's weights:")
for i in range(n_neurons):
    a_i, z_i = single_neuron(x, W[i], b[i])
    print(f"  Neuron {i}: z={z_i:.4f}, a={a_i:.4f}  |  "
          f"Layer z[{i}]={z[i]:.4f}, a[{i}]={a[i]:.4f}  |  "
          f"Match: {np.isclose(a_i, a[i])}")

In [ ]:
# Stacking two layers into a simple network
np.random.seed(42)

# Network: 3 inputs -> 4 hidden -> 1 output
x = np.array([0.5, -0.3, 0.8])

# Hidden layer (4 neurons)
W1 = np.random.randn(4, 3)
b1 = np.random.randn(4)

# Output layer (1 neuron)
W2 = np.random.randn(1, 4)
b2 = np.random.randn(1)

# Forward pass
h, z1 = dense_layer(x, W1, b1, activation_fn=sigmoid)     # hidden layer
y, z2 = dense_layer(h, W2, b2, activation_fn=sigmoid)     # output layer

print(f"Input:          {x}")
print(f"Hidden output:  {h}")
print(f"Network output: {y}")
print()
print("This is a complete forward pass through a 2-layer network!")

---
## 9. Exercise: Different Activations

**Task:** Modify the single neuron to use different activation functions and observe how the output changes.

1. Implement `relu(z)` and `tanh_activation(z)` functions
2. Run the same neuron (same weights, bias, input) with sigmoid, ReLU, and tanh
3. Compare the outputs -- which activation gives the largest output? The smallest?

```python
# Your code here
def relu(z):
    # TODO: implement ReLU
    pass

def tanh_activation(z):
    # TODO: implement tanh
    pass

# Test with the same input, weights, bias
x = np.array([0.5, -0.3, 0.8])
w = np.array([0.4967, -0.1383, 0.6477])  # fixed weights
b = 0.5

# TODO: compute output with each activation and print results
```

In [ ]:
# Try the exercise yourself before scrolling down!











### Solution

In [ ]:
# ----- Solution -----

def relu(z):
    """ReLU activation: max(0, z)"""
    return np.maximum(0, z)


def tanh_activation(z):
    """Tanh activation."""
    return np.tanh(z)


# Fixed input, weights, bias
x = np.array([0.5, -0.3, 0.8])
w = np.array([0.4967, -0.1383, 0.6477])
b = 0.5

activations = {
    'Sigmoid': sigmoid,
    'ReLU': relu,
    'Tanh': tanh_activation,
}

z_val = np.dot(w, x) + b
print(f"Pre-activation z = {z_val:.4f}")
print(f"{'Activation':<12} {'Output':>10}")
print("-" * 24)

for name, fn in activations.items():
    a_val, _ = single_neuron(x, w, b, activation_fn=fn)
    print(f"{name:<12} {a_val:>10.4f}")

print()
print("Observations:")
print("- Sigmoid squashes output to (0, 1) -> moderate value")
print("- ReLU passes positive z unchanged, clips negative to 0")
print("- Tanh squashes output to (-1, 1) -> centered around zero")

---
## 10. Common Mistakes & Debugging Tips

**1. Shape mismatches in matrix multiplication**
- If $\mathbf{W}$ is $(m, n)$ and $\mathbf{x}$ is $(n,)$, the result is $(m,)$. Ensure dimensions align.
- Tip: Always print `.shape` when debugging.

**2. Forgetting the bias**
- Without bias, the neuron's output is forced through the origin, severely limiting expressiveness.
- Always include $b$ unless you have a specific reason not to.

**3. Using no activation (or only linear activations)**
- Without nonlinear activations, a multi-layer network collapses into a single linear transformation: $\mathbf{W}_2(\mathbf{W}_1 \mathbf{x} + \mathbf{b}_1) + \mathbf{b}_2 = \mathbf{W}' \mathbf{x} + \mathbf{b}'$.
- Always use nonlinear activations in hidden layers.

**4. Confusing weight matrix dimensions**
- Convention: $\mathbf{W} \in \mathbb{R}^{\text{n\_outputs} \times \text{n\_inputs}}$ so that $\mathbf{W} \mathbf{x}$ has the right shape.
- Some frameworks (e.g., PyTorch `nn.Linear`) store $\mathbf{W}$ as `(out_features, in_features)` but accept input as `(batch, in_features)` and compute $\mathbf{x} \mathbf{W}^T$.

**5. Numerical overflow in sigmoid**
- For very large negative $z$, `np.exp(-z)` can overflow. Use `scipy.special.expit` or clip $z$ to a safe range.
- Alternatively: `sigmoid(z) = np.where(z >= 0, 1/(1+np.exp(-z)), np.exp(z)/(1+np.exp(z)))`

---

**Next notebook:** [02 - Forward Propagation and Activations](02_Forward_Propagation_and_Activations.ipynb) -- we dive deeper into activation functions, matrix-form forward propagation, and practical guidelines.